# 04 — Real Environmental Audio

## Kaynak ve WAV

InspectorJ kuş kaydı, CC-BY-4.0. İlk çalıştırmada OGG indirilir, SHA-256 kontrol edilir ve WAV kopyası oluşturulur. Ses dosyaları Git dışındadır. Örnekleme hızı korunur.

In [ ]:
from pathlib import Path
import sys, json, csv, hashlib, platform, importlib.metadata
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
sys.path.insert(0,str(ROOT/"src"))
from audio.real_sample import load_robin, ATTRIBUTION, SHA256
from audio.features import frame_features, pitch_features
EXPERIMENT_ID = "04_real_audio"
FIGURES = ROOT/"results"/"figures"/EXPERIMENT_ID
TABLES = ROOT/"results"/"tables"/EXPERIMENT_ID
EXPERIMENT = ROOT/"experiments"/EXPERIMENT_ID
for folder in [FIGURES,TABLES,EXPERIMENT]: folder.mkdir(parents=True,exist_ok=True)
y,sr,wav = load_robin(ROOT)
print(ATTRIBUTION)
print(f"sr={sr} Hz, duration={len(y)/sr:.4f} s")
display(Audio(y,rate=sr,normalize=False))


## RMS ve centroid

2048 örneklik frame, 256 örneklik hop; RMS dikdörtgen, centroid Hann pencere kullanır. −50 dBFS eşiği bu kayıtta düşük seviyeli frame’leri maskelemek için bir analiz seçimidir. Otomatik olay sınırı veya makale protokolü değildir.

In [ ]:
settings={"frame_length":2048,"hop_length":256,"threshold_db":-50}
features=frame_features(y,sr,**settings)
active=features["active"]
metrics={"sr":int(sr),"duration_s":len(y)/sr,"frames":len(active),
         "active_frames":int(active.sum()),"global_rms":float(np.sqrt(np.mean(y**2))),
         "mean_centroid_active_hz":float(np.mean(features["centroid_hz"][active])) if active.any() else None}
fig,axes=plt.subplots(3,1,figsize=(11,8),sharex=True)
stride=max(1,len(y)//3000)
axes[0].plot((np.arange(len(y))/sr)[::stride],y[::stride]);axes[0].set(ylabel="Genlik")
axes[1].plot(features["time_s"],features["rms_dbfs"]);axes[1].set(ylabel="RMS (dBFS ref=1)")
axes[1].axhline(-50,color="gray",linestyle="--")
axes[2].plot(features["time_s"],features["centroid_hz"]);axes[2].set(ylabel="Centroid (Hz)",xlabel="Zaman (s)")
plt.tight_layout();fig.savefig(FIGURES/"real_audio_features.svg",bbox_inches="tight");plt.show()


## Çıktılar

CSV frame değerlerini, JSON veri kaynağını, ayarları ve sürümleri kaydeder. Centroid’in yüksek olması tek başına yüksek pitch veya kaliteli ses anlamına gelmez.

In [ ]:
with (TABLES/"frame_features.csv").open("w",newline="",encoding="utf-8") as f:
    writer=csv.writer(f)
    writer.writerow(features.keys())
    for values in zip(*features.values()):
        writer.writerow([v.item() if isinstance(v,np.generic) and np.isfinite(v) else ("" if isinstance(v,(float,np.floating)) and not np.isfinite(v) else v) for v in values])
run={"experiment_id":EXPERIMENT_ID,"time_utc":datetime.now(timezone.utc).isoformat(),
     "source":ATTRIBUTION,"source_ogg_sha256":SHA256,"python":platform.python_version(),
     "platform":platform.system(),"settings":settings,"metrics":metrics,
     "packages":{p:importlib.metadata.version(p) for p in ["numpy","librosa","soundfile","matplotlib"]}}
(EXPERIMENT/"run.json").write_text(json.dumps(run,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
print(metrics)
